# param-grad-access — ex2: compute per-parameter gradient L2 norms with None-grad skip

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `param-grad-access`. Running the final beacon cell reports progress against the `PyTorch: param.grad access` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: param.grad access` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`param-grad-access`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "param-grad-access"
DD_SUBTOPIC = "PyTorch: param.grad access"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## param.grad — gradient norm logging

Per-parameter gradient norms are the first-line training diagnostic. They let you spot:

- **vanishing grads** (norm → 0 in deep layers — model not learning)
- **exploding grads** (norm → ∞ — need clipping / smaller lr)
- **dead layers** (grad is `None` — that subgraph wasn't reached)

```python
for name, p in model.named_parameters():
    if p.grad is None:
        print(f'{name}: NO GRAD')
        continue
    print(f'{name}: {p.grad.norm(p=2).item():.4e}')
```

The previous drill (ex1) applied the canonical SGD step with the None-grad skip rule. This drill targets the **diagnostic** half: computing the L2 norm of each parameter's gradient (skipping None) and returning a dict keyed by param NAME — exactly the data structure logged to TensorBoard.

### Exercise 2 — compute per-parameter gradient L2 norms with None-grad skip

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `named_parameters()` + `.grad.norm()` pattern to build a diagnostic dict mapping parameter name → L2 grad norm, omitting params whose `.grad` is `None`.
> Keywords: grad-norm, named-parameters, diagnostic, logging
> ```

**KCs targeted:** `param-grad-access`, `param-grad-none-guard-skip`

Implement `ex2_grad_norms(model)`. Return a `dict[str, float]` mapping each parameter's NAME (from `model.named_parameters()`) to `p.grad.norm(p=2).item()`. Parameters whose `.grad is None` must be **OMITTED** from the dict (not present as None, not present as 0.0 — absent entirely).

**Required pattern.**
```python
out = {}
for name, p in model.named_parameters():
    if p.grad is None:
        continue
    out[name] = p.grad.norm(p=2).item()
return out
```

Inputs:
- `model`: any `nn.Module` whose `.named_parameters()` yields `(str, Parameter)` pairs.

Output:
- `dict[str, float]` — names are stable across calls (PyTorch names them by attribute path, e.g. `'fc1.weight'`, `'fc1.bias'`).

**Why per-parameter not flat-concat.** A flat concat (`.norm()` on every grad joined together) hides per-layer behavior. The per-param dict is what lets you spot the dead layer.

In [ ]:
def ex2_grad_norms(model) -> dict:
    """{param-name: L2 grad norm} dict, skipping params whose .grad is None."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # Build a tiny model with named parameters spanning a Linear + a free Parameter.
    class Mini(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc = nn.Linear(3, 2)
            self.bias_extra = nn.Parameter(t.zeros(2))
        def forward(self, x):
            return self.fc(x) + self.bias_extra

    model = Mini()
    x = t.tensor([[1.0, 1.0, 1.0]])
    y = t.tensor([[3.0, 5.0]])
    loss = ((model(x) - y) ** 2).sum()
    loss.backward()

    norms = ex2_grad_norms(model)
    assert isinstance(norms, dict), f'must return dict, got {type(norms)}'
    # Every param participated → all three names must be in the dict.
    assert set(norms.keys()) == {'fc.weight', 'fc.bias', 'bias_extra'}, (
        f'expected keys {{fc.weight, fc.bias, bias_extra}}, got {set(norms.keys())}'
    )
    # Values are floats and match the manual computation.
    for name, p in model.named_parameters():
        expected = p.grad.norm(p=2).item()
        got = norms[name]
        assert isinstance(got, float), f'{name}: value must be Python float, got {type(got)}'
        assert abs(got - expected) < 1e-6, f'{name}: got {got}, expected {expected}'

    # All norms strictly positive (every param had a non-trivial gradient here).
    for name, n in norms.items():
        assert n > 0, f'{name}: norm should be > 0 after non-trivial backward, got {n}'

    # --- None-grad skip behavior ---
    # Fresh model, NO backward called — every param has grad=None.
    fresh = Mini()
    empty = ex2_grad_norms(fresh)
    assert empty == {}, f'fresh model with no backward must yield empty dict, got {empty}'

    # Mixed — manually populate ONE param's grad, leave the others None.
    mixed = Mini()
    mixed.fc.weight.grad = t.ones_like(mixed.fc.weight)  # norm = sqrt(6)
    got = ex2_grad_norms(mixed)
    assert set(got.keys()) == {'fc.weight'}, f'only fc.weight should appear; got {set(got.keys())}'
    assert abs(got['fc.weight'] - (6.0 ** 0.5)) < 1e-5, f'norm of all-ones (3,2) tensor is sqrt(6); got {got[chr(39)+"fc.weight"+chr(39)]}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_grad_norms(model) -> dict:
    out = {}
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        out[name] = p.grad.norm(p=2).item()
    return out
```

**Why `named_parameters()` not `parameters()`.** The names give you the path through the module tree (`fc1.weight`, `block.0.attn.qkv`, etc.) — required for logging to TensorBoard / Weights & Biases per-parameter charts. `parameters()` yields tensors only, with no name attached.

**The skip vs the zero.** Omitting a param entirely (vs storing `0.0`) is the right call here: it tells the consumer 'this param didn't get a gradient', not 'this param got a gradient of magnitude exactly zero' — two semantically different conditions that look identical in a bar chart if you collapse them.

**Difference from ex1.** ex1 PERFORMED the SGD step (`p.data -= lr * p.grad`) with the None-grad skip. ex2 only READS the gradient (`p.grad.norm()`) with the same skip rule — the diagnostic counterpart of the update step.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()